# SAE Operational Fingerprint — Train from Scratch on Gemma-3-1B

**Model:** `google/gemma-3-1b-pt`  
**SAE:** Trained from scratch on operation-specific activations (not reused)  
**Task:** Indirect Object Identification (IOI)

## Why train from scratch?

A general-purpose SAE is trained on web text and learns features for *everything*.
We train the SAE only on activations collected during task execution.
The dictionary it learns is shaped by the operation — so if it finds structure,
that structure is operational by construction, not a side effect of general pretraining.

## Pipeline
```
1. Collect activations  →  Gemma-3-1B on IOI + CTRL prompts, layer 17 residual stream
2. Train SAE            →  TopK SAE on collected activations (offline)
3. Fingerprint analysis →  AUC, format invariance, constant-trace, PCA
```

⚠️ Gemma is gated. Accept terms at https://huggingface.co/google/gemma-3-1b-pt  
then set `HF_TOKEN` below.

In [ ]:
!pip install -q transformer_lens

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
HF_TOKEN   = "hf_..."    # your HuggingFace token
MODEL_NAME = "google/gemma-3-1b-pt"
LAYER      = 17          # residual stream hook; Gemma-3-1B has 26 layers

# SAE hyperparameters
SAE_K      = 32          # TopK: number of active features per token
SAE_RATIO  = 8           # dictionary expansion: d_sae = d_model × SAE_RATIO
SAE_LR     = 3e-4
SAE_EPOCHS = 10
SAE_BATCH  = 256

# Data
N_PROMPTS  = 300         # per condition (IOI and CTRL); reduce to 150 if OOM

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import itertools, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from huggingface_hub import login
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from scipy.stats import spearmanr
from numpy.linalg import norm
from transformer_lens import HookedTransformer
from tqdm.auto import trange

login(token=HF_TOKEN, add_to_git_credential=False)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

In [ ]:
# ── Load model ────────────────────────────────────────────────────────────────
model = HookedTransformer.from_pretrained(
    MODEL_NAME,
    center_writing_weights=False,
    dtype="bfloat16",
)
model.eval().to(DEVICE)

D_MODEL = model.cfg.d_model
D_SAE   = D_MODEL * SAE_RATIO
HOOK    = f"blocks.{LAYER}.hook_resid_pre"
print(f"d_model={D_MODEL}  d_sae={D_SAE}  hook={HOOK}")

In [ ]:
# ── Prompt generation ─────────────────────────────────────────────────────────
NAMES   = [("Mary","John"),("Alice","Bob"),("Sarah","Mike"),("Emma","James"),
           ("Lily","Tom"),("Grace","Paul"),("Claire","Mark"),("Sophie","Luke"),("Anna","Chris")]
VERBS   = ["went","traveled","walked","drove","rushed","headed"]
PLACES  = ["store","park","school","office","market","library","cafe"]
OBJECTS = ["drink","bag","book","note","gift","key","pen"]
CTRL_N  = ["Laura","David","Zoe","Ben","Iris","Owen"]

def ioi_prompt(io, s, v, pl, ob):
    return f"When {io} and {s} {v}ed to the {pl}, {s} gave a {ob} to"

def ctrl_prompt(a, b, c, v, pl, ob):
    return f"When {a} and {b} {v}ed to the {pl}, {c} gave a {ob} to"

rng = random.Random(42)
ioi_data, ctrl_data = [], []

for (io, s) in NAMES:
    for v, pl, ob in itertools.product(VERBS[:4], PLACES[:4], OBJECTS[:4]):
        ioi_data.append(dict(prompt=ioi_prompt(io,s,v,pl,ob),
                             io=io, s=s, v=v, pl=pl, ob=ob, task="IOI", correct=io))

for (a, b) in NAMES[:6]:
    c = rng.choice(CTRL_N)
    for v, pl, ob in itertools.product(VERBS[:4], PLACES[:4], OBJECTS[:4]):
        ctrl_data.append(dict(prompt=ctrl_prompt(a,b,c,v,pl,ob),
                              io=a, s=b, ctrl=c, v=v, pl=pl, ob=ob, task="CTRL", correct=None))

N = min(len(ioi_data), len(ctrl_data), N_PROMPTS)
rng.shuffle(ioi_data); rng.shuffle(ctrl_data)
ioi_data, ctrl_data = ioi_data[:N], ctrl_data[:N]
print(f"IOI: {len(ioi_data)}   CTRL: {len(ctrl_data)}")
print("Sample:", ioi_data[0]["prompt"], "→", ioi_data[0]["correct"])

In [ ]:
# ── Step 1: Collect activations ───────────────────────────────────────────────
# We collect residual stream vectors at the *last token* position.
# These are the raw activations the SAE will be trained to reconstruct.

@torch.no_grad()
def collect_activations(records, batch_size=8):
    """Returns (N, d_model) float32 tensor of last-token residual stream."""
    all_acts = []
    prompts  = [r["prompt"] for r in records]
    for i in range(0, len(prompts), batch_size):
        batch  = prompts[i:i+batch_size]
        tokens = model.to_tokens(batch, prepend_bos=True)
        _, cache = model.run_with_cache(tokens, names_filter=HOOK)
        acts = cache[HOOK][:, -1, :].to(torch.float32).cpu()
        all_acts.append(acts)
    return torch.cat(all_acts, dim=0)

print("Collecting IOI activations...")
ioi_acts  = collect_activations(ioi_data)
print("Collecting CTRL activations...")
ctrl_acts = collect_activations(ctrl_data)

# Training corpus: all activations together (SAE sees both conditions)
train_acts = torch.cat([ioi_acts, ctrl_acts], dim=0)
print(f"Training corpus: {train_acts.shape}  (mean={train_acts.mean():.4f} std={train_acts.std():.4f})")

In [ ]:
# ── Step 2: TopK SAE ──────────────────────────────────────────────────────────
# Architecture: x → pre-bias → W_enc → TopK → features → W_dec → x_hat
# Loss: MSE reconstruction + auxiliary dead-feature penalty
#
# TopK instead of L1:
#   - Fixed sparsity (k active features per sample) → stable, no λ tuning
#   - Cleaner comparison across conditions (always same k)
#   - Dead features avoided via auxiliary loss (Anthropic TopK SAE recipe)

class TopKSAE(nn.Module):
    def __init__(self, d_in: int, d_sae: int, k: int):
        super().__init__()
        self.k     = k
        self.d_sae = d_sae
        self.pre_bias = nn.Parameter(torch.zeros(d_in))
        self.W_enc    = nn.Parameter(torch.empty(d_in, d_sae))
        self.b_enc    = nn.Parameter(torch.zeros(d_sae))
        self.W_dec    = nn.Parameter(torch.empty(d_sae, d_in))
        self.b_dec    = nn.Parameter(torch.zeros(d_in))
        nn.init.kaiming_uniform_(self.W_enc)
        # Decoder columns initialised to unit norm
        nn.init.kaiming_uniform_(self.W_dec)
        with torch.no_grad():
            self.W_dec.data = nn.functional.normalize(self.W_dec.data, dim=1)

    def encode(self, x):
        x_centered = x - self.pre_bias
        pre_acts   = x_centered @ self.W_enc + self.b_enc   # (B, d_sae)
        # TopK: keep only k largest activations, zero the rest
        topk_vals, topk_idx = pre_acts.topk(self.k, dim=-1)
        acts = torch.zeros_like(pre_acts)
        acts.scatter_(-1, topk_idx, topk_vals.clamp(min=0))
        return acts

    def decode(self, acts):
        return acts @ self.W_dec + self.b_dec

    def forward(self, x):
        acts  = self.encode(x)
        x_hat = self.decode(acts)
        return acts, x_hat

    @torch.no_grad()
    def normalise_decoder(self):
        """Keep decoder columns unit-norm (called after each optimizer step)."""
        self.W_dec.data = nn.functional.normalize(self.W_dec.data, dim=1)


sae = TopKSAE(D_MODEL, D_SAE, SAE_K).to(DEVICE)
print(f"SAE: d_in={D_MODEL}  d_sae={D_SAE}  k={SAE_K}  params={sum(p.numel() for p in sae.parameters()):,}")

In [ ]:
# ── Step 3: Train SAE ─────────────────────────────────────────────────────────
# Main loss:  MSE(x, x_hat)
# Aux loss:   encourage dead features to explain residual (Anthropic recipe)
# The aux loss is weighted low — it's a regulariser, not the main objective.

AUX_WEIGHT  = 1/32   # weight on auxiliary dead-feature loss
DEAD_THRESH = 1e-4   # feature is "dead" if mean activation below this

optimizer = torch.optim.Adam(sae.parameters(), lr=SAE_LR, betas=(0.9, 0.999))
dataset   = torch.utils.data.TensorDataset(train_acts)
loader    = torch.utils.data.DataLoader(dataset, batch_size=SAE_BATCH, shuffle=True)

# Track feature usage for dead-feature detection
feature_usage = torch.zeros(D_SAE, device=DEVICE)

history = []
for epoch in range(SAE_EPOCHS):
    epoch_loss = epoch_mse = epoch_aux = 0.0
    for (x_batch,) in loader:
        x = x_batch.to(DEVICE)

        acts, x_hat = sae(x)

        # Update running feature usage
        with torch.no_grad():
            feature_usage = 0.99 * feature_usage + 0.01 * (acts > 0).float().mean(dim=0)

        mse = (x - x_hat).pow(2).mean()

        # Auxiliary loss: dead features reconstruct the residual
        dead_mask = (feature_usage < DEAD_THRESH).float()   # (d_sae,)
        if dead_mask.sum() > 0:
            residual   = (x - x_hat).detach()
            # Project residual onto dead decoder columns
            dead_recon = (acts * dead_mask) @ sae.W_dec
            aux        = (residual - dead_recon).pow(2).mean()
        else:
            aux = torch.tensor(0.0, device=DEVICE)

        loss = mse + AUX_WEIGHT * aux

        optimizer.zero_grad()
        loss.backward()
        # Gradient clipping for stability
        nn.utils.clip_grad_norm_(sae.parameters(), max_norm=1.0)
        optimizer.step()
        sae.normalise_decoder()

        epoch_loss += loss.item()
        epoch_mse  += mse.item()
        epoch_aux  += aux.item()

    n_dead = (feature_usage < DEAD_THRESH).sum().item()
    n_steps = len(loader)
    row = dict(epoch=epoch+1,
               loss=epoch_loss/n_steps,
               mse=epoch_mse/n_steps,
               aux=epoch_aux/n_steps,
               dead=n_dead)
    history.append(row)
    print(f"Epoch {epoch+1:2d}/{SAE_EPOCHS}  "
          f"loss={row['loss']:.4f}  mse={row['mse']:.4f}  aux={row['aux']:.4f}  "
          f"dead={n_dead}/{D_SAE}")

df_hist = pd.DataFrame(history)

In [ ]:
# ── Training diagnostics ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(df_hist["epoch"], df_hist["mse"], marker="o")
axes[0].set(title="Reconstruction MSE", xlabel="Epoch")

axes[1].plot(df_hist["epoch"], df_hist["dead"], marker="o", color="tomato")
axes[1].set(title=f"Dead features (/{D_SAE})", xlabel="Epoch")

# Feature activation frequency histogram
freq = feature_usage.cpu().numpy()
axes[2].hist(freq[freq > DEAD_THRESH], bins=50, color="steelblue")
axes[2].set(title="Feature usage distribution (live features)", xlabel="Mean activation frequency")

plt.tight_layout(); plt.savefig("sae_training.png", dpi=150); plt.show()

# Variance explained
with torch.no_grad():
    sample   = train_acts[:500].to(DEVICE)
    _, recon = sae(sample)
    var_expl = 1 - (sample - recon).pow(2).sum() / sample.pow(2).sum()
print(f"\nVariance explained by SAE: {var_expl.item():.2%}")
print(f"Live features: {(feature_usage > DEAD_THRESH).sum().item()}/{D_SAE}")

In [ ]:
# ── Extract SAE features for IOI and CTRL ─────────────────────────────────────
sae.eval()
with torch.no_grad():
    ioi_feats  = sae.encode(ioi_acts.to(DEVICE)).cpu().numpy()
    ctrl_feats = sae.encode(ctrl_acts.to(DEVICE)).cpu().numpy()

print(f"IOI features:  {ioi_feats.shape}  sparsity: {(ioi_feats==0).mean():.2%} zeros")
print(f"CTRL features: {ctrl_feats.shape}  sparsity: {(ctrl_feats==0).mean():.2%} zeros")

In [ ]:
# ── Accuracy ──────────────────────────────────────────────────────────────────
@torch.no_grad()
def measure_accuracy(records, batch_size=8):
    results = []
    for i in range(0, len(records), batch_size):
        batch  = records[i:i+batch_size]
        tokens = model.to_tokens([r["prompt"] for r in batch], prepend_bos=True)
        logits = model(tokens)[:, -1, :].float()
        for j, rec in enumerate(batch):
            pred = model.tokenizer.decode([logits[j].argmax().item()]).strip()
            results.append(pred == rec["correct"] or pred == " " + rec["correct"])
    return results

ioi_correct = measure_accuracy(ioi_data)
acc = np.mean(ioi_correct)
print(f"IOI accuracy: {acc:.2%}")

df_ioi = pd.DataFrame(ioi_data)
df_ioi["correct_bool"] = ioi_correct
df_ioi["feat_idx"]     = range(len(df_ioi))
print(df_ioi.groupby(["io","s"])["correct_bool"].mean().to_string())

In [ ]:
# ── Q1: Linear AUC — does the fingerprint exist? ─────────────────────────────
X = np.concatenate([ioi_feats, ctrl_feats], axis=0)
y = np.array([1]*len(ioi_feats) + [0]*len(ctrl_feats))
clf  = LogisticRegression(max_iter=1000, C=0.1)
aucs = cross_val_score(clf, X, y, cv=5, scoring="roc_auc")
print(f"Linear AUC (IOI vs CTRL):  {aucs.mean():.3f} ± {aucs.std():.3f}")
print("AUC > 0.7 → operational fingerprint is linearly readable from custom SAE features")

In [ ]:
# ── Q2: Format invariance ─────────────────────────────────────────────────────
def cosine(a, b): return float(np.dot(a,b) / (norm(a)*norm(b) + 1e-8))

pair_centroids = {}
for (io,s), grp in df_ioi.groupby(["io","s"]):
    pair_centroids[(io,s)] = ioi_feats[grp["feat_idx"].values].mean(axis=0)

within_sims = []
for (io,s), grp in df_ioi.groupby(["io","s"]):
    feats = ioi_feats[grp["feat_idx"].values]
    for i in range(len(feats)):
        for j in range(i+1, min(i+8, len(feats))):
            within_sims.append(cosine(feats[i], feats[j]))

pairs = list(pair_centroids.items())
between_sims = [cosine(pairs[i][1], pairs[j][1])
                for i in range(len(pairs)) for j in range(i+1, len(pairs))]

ioi_c  = ioi_feats.mean(axis=0)
ctrl_c = ctrl_feats.mean(axis=0)
cross  = cosine(ioi_c, ctrl_c)

print("Format invariance (cosine similarity):")
print(f"  Within  name-pair (vary surface):   {np.mean(within_sims):.4f} ± {np.std(within_sims):.4f}")
print(f"  Between name-pairs (same IOI task): {np.mean(between_sims):.4f} ± {np.std(between_sims):.4f}")
print(f"  IOI centroid vs CTRL centroid:      {cross:.4f}")
print()
print("Ideal: within ≈ between >> cross")
print("       → operation-specific, not name-specific")

In [ ]:
# ── Q3: Constant-trace metric ─────────────────────────────────────────────────
active_mask = ioi_feats.max(axis=0) > 0

pair_var, pair_acc, pair_labels = [], [], []
for (io,s), grp in df_ioi.groupby(["io","s"]):
    feats = ioi_feats[grp["feat_idx"].values]
    v = feats[:, active_mask].var(axis=0).mean()
    pair_var.append(float(v))
    pair_acc.append(grp["correct_bool"].mean())
    pair_labels.append(f"{io}/{s}")

rho, pval = spearmanr(pair_var, pair_acc)
print(f"Spearman ρ(variance, accuracy):  {rho:.3f}  p={pval:.4f}")
print("Hypothesis: ρ < 0  (flatter trace → higher accuracy)")

# Correct vs incorrect
c_idx = np.where(np.array(ioi_correct))[0]
w_idx = np.where(~np.array(ioi_correct))[0]
print(f"\nCorrect: {len(c_idx)}   Incorrect: {len(w_idx)}")
if len(c_idx) > 5 and len(w_idx) > 5:
    vc = ioi_feats[c_idx][:, active_mask].var(axis=0).mean()
    vw = ioi_feats[w_idx][:, active_mask].var(axis=0).mean()
    sc = cosine(ioi_feats[c_idx].mean(axis=0), ioi_c)
    sw = cosine(ioi_feats[w_idx].mean(axis=0), ioi_c)
    print(f"  Variance  correct={vc:.5f}   incorrect={vw:.5f}")
    print(f"  Cosine-to-centroid  correct={sc:.4f}   incorrect={sw:.4f}")

fig, ax = plt.subplots(figsize=(6,4))
ax.scatter(pair_var, pair_acc, s=80)
for i, lbl in enumerate(pair_labels):
    ax.annotate(lbl, (pair_var[i], pair_acc[i]),
                textcoords="offset points", xytext=(4,3), fontsize=7)
ax.set(xlabel="Mean feature variance (active)", ylabel="Accuracy",
       title=f"Constant-trace  ρ={rho:.3f}  p={pval:.3f}")
plt.tight_layout(); plt.savefig("ioi_variance.png", dpi=150); plt.show()

In [ ]:
# ── Q4: PCA ───────────────────────────────────────────────────────────────────
X_all = np.concatenate([ioi_feats, ctrl_feats], axis=0)
labels_task = ["IOI"]*len(ioi_feats) + ["CTRL"]*len(ctrl_feats)
labels_pair = [f"{r['io']}/{r['s']}" for r in ioi_data] + ["ctrl"]*len(ctrl_data)
pcs = PCA(n_components=2).fit_transform(StandardScaler().fit_transform(X_all))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14,5))

for task, color in [("IOI","steelblue"),("CTRL","tomato")]:
    m = np.array(labels_task)==task
    ax1.scatter(pcs[m,0], pcs[m,1], c=color, alpha=0.35, s=18, label=task)
ax1.legend(); ax1.set_title("PCA: IOI vs CTRL (custom SAE)")

ioi_mask  = np.array(labels_task)=="IOI"
ioi_pairs = np.array(labels_pair)[ioi_mask]
unique_pairs = sorted(set(ioi_pairs))
cmap = plt.cm.get_cmap("tab10", len(unique_pairs))
for i, p in enumerate(unique_pairs):
    m = ioi_pairs==p
    ax2.scatter(pcs[ioi_mask][m,0], pcs[ioi_mask][m,1],
                c=[cmap(i)], alpha=0.5, s=18, label=p)
ax2.legend(fontsize=6, ncol=2); ax2.set_title("PCA: IOI name-pairs")

plt.tight_layout(); plt.savefig("ioi_pca.png", dpi=150); plt.show()

In [ ]:
# ── Q5: Top discriminating SAE features ───────────────────────────────────────
TOPK  = 20
delta = ioi_feats.mean(axis=0) - ctrl_feats.mean(axis=0)
top   = np.argsort(delta)[::-1][:TOPK]
print(f"Top {TOPK} features more active in IOI vs CTRL:")
print(f"{'Feature':>8}  {'IOI mean':>10}  {'CTRL mean':>10}  {'Δ':>10}")
im, cm_ = ioi_feats.mean(axis=0), ctrl_feats.mean(axis=0)
for idx in top:
    print(f"{idx:>8}  {im[idx]:>10.5f}  {cm_[idx]:>10.5f}  {delta[idx]:>10.5f}")

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
print("=" * 65)
print("CUSTOM SAE — IOI FINGERPRINT SUMMARY")
print("=" * 65)
print(f"  Model:              {MODEL_NAME}  layer {LAYER}")
print(f"  SAE:                trained from scratch  d={D_SAE}  k={SAE_K}")
print(f"  Variance explained: {var_expl.item():.2%}")
print(f"  Live features:      {(feature_usage>DEAD_THRESH).sum().item()}/{D_SAE}")
print()
print(f"  IOI accuracy:       {acc:.2%}")
print(f"  Linear AUC:         {aucs.mean():.3f} ± {aucs.std():.3f}")
print(f"  Within-pair sim:    {np.mean(within_sims):.4f}")
print(f"  Between-pair sim:   {np.mean(between_sims):.4f}")
print(f"  IOI vs CTRL sim:    {cross:.4f}")
print(f"  Variance-acc ρ:     {rho:.3f}  p={pval:.4f}")
print()
print("DECISIONS")
ok_auc = aucs.mean() > 0.7
ok_fmt = np.mean(within_sims) > np.mean(between_sims)
ok_rho = rho < -0.2
print(f"  AUC > 0.7?         {'✓' if ok_auc else '✗'}  (fingerprint {'found' if ok_auc else 'absent — try k or layer'})")
print(f"  within > between?  {'✓' if ok_fmt else '✗'}  (format-{'invariant' if ok_fmt else 'sensitive'})")
print(f"  ρ < −0.2?          {'✓' if ok_rho else '✗'}  (constant-trace {'predictive' if ok_rho else 'not predictive'})")
print()
if ok_auc and ok_fmt:
    print("→ PROCEED: add a second operation (e.g. greater-than) and test RSA compositionality (Idea C)")
elif ok_auc:
    print("→ Fingerprint found but name-specific. Increase name diversity or try k=16.")
else:
    print("→ No fingerprint. Try: different layer, larger k, more training epochs.")
print("=" * 65)